# Kouman Edge 1B — Quantification GGUF Complète (`Q8_0 → Q2_K` + `imatrix` 4 Langues) sur Google Colab

Notebook **100 % autonome** (fonctionne sur **Google Colab gratuit CPU ou T4** en ~6 à 8 minutes) :
1. Télécharge ton modèle **Instruct 1B BF16** déjà fusionné depuis **`Tree-AI-lab/kouman-edge-1b`**.
2. Compile **`llama.cpp`** et convertit le modèle en GGUF `F16` maître (`~2,31 Go`).
3. Calibre une **Importance Matrix (`imatrix`)** sur nos **4 langues** (**Baoulé, Dioula, Français, Anglais**) pour protéger la qualité en `Q4` / `Q3` / `Q2`.
4. Génère les **10 fichiers GGUF de `Q8_0` jusqu'à `Q2_K`**.
5. Teste l'inférence réelle sur **`Q4_K_M`** et **`Q2_K`**, crée la **Model Card (`README.md`)**, et ajoute l'ensemble directement dans **`Tree-AI-lab/kouman-edge-1b`** (sans toucher au modèle Instruct BF16 existant).

## 1. Dépendances, Configuration & Authentification Hugging Face

In [ ]:
%pip install --quiet --upgrade pip
%pip install --quiet "huggingface_hub>=0.34.0,<1.0.0" "gguf" "pyarrow>=17.0.0" "sentencepiece" "protobuf" "safetensors" "transformers>=4.45.0,<5.0.0"

In [ ]:
import getpass
import os
import shutil
import subprocess
import sys
from pathlib import Path

os.environ["HF_HUB_DISABLE_XET"] = "1"
try:
    import huggingface_hub.constants as _hf_const
    import huggingface_hub._commit_api as _hf_commit
    _hf_const.HF_HUB_DISABLE_XET = True
    if hasattr(_hf_commit, "is_xet_available"):
        _hf_commit.is_xet_available = lambda: False
except Exception:
    pass

import pyarrow.parquet as pq
from huggingface_hub import HfApi, get_token, hf_hub_download, login, snapshot_download

# Option A : colle directement ton token hf_... ci-dessous (recommandé sur Colab)
# Option B : laisse vide pour ouvrir le widget officiel Hugging Face (bouton "Login")
JETON_HF_MANUEL = ""  # @param {type:"string"}

JETON_HF = JETON_HF_MANUEL.strip()
if not JETON_HF:
    login(add_to_git_credential=False)
    JETON_HF = (get_token() or "").strip()
else:
    login(token=JETON_HF, add_to_git_credential=False)

if not JETON_HF:
    raise PermissionError("[ERREUR] Connexion Hugging Face annulée : colle ton token hf_... dans JETON_HF_MANUEL.")
os.environ["HF_TOKEN"] = JETON_HF

DEPOT_MODELE = "Tree-AI-lab/kouman-edge-1b"
DEPOT_DATASET_CALIB = "issoufzousko07/kouman-edge-1b-sft-v4-5-3k"
NOM_COURT = "kouman-edge-1b"

UTILISER_IMATRIX = True       # Protège le Baoulé et le Dioula sur Q4 / Q3 / Q2
CHUNKS_IMATRIX = 30           # Rapide sur Colab CPU (~2 min) et suffisant pour 1B
GARDER_GGUF_F16 = False       # Mettre True si tu veux aussi uploader le F16 (2,31 Go) sur HF
PUBLIER_SUR_HF = True

QUANTIFICATIONS_GGUF = [
    ("Q8_0", "Quasi sans perte (desktop / flagship 8+ Go RAM)", False),
    ("Q6_K", "Qualité quasi-BF16, excellent compromis", True),
    ("Q5_K_M", "Haute qualité (smartphones haut de gamme / PC)", True),
    ("Q5_K_S", "Haute qualité compacte", True),
    ("Q4_K_M", "Recommandé mobile Android / iOS (équilibre qualité/vitesse)", True),
    ("Q4_K_S", "Mobile compact 4 bits", True),
    ("Q4_0", "Mobile ultra-rapide CPU ARM (NEON / i8mm)", True),
    ("Q3_K_M", "Smartphones entrée de gamme (4 Go RAM)", True),
    ("Q3_K_S", "Très compact 3 bits", True),
    ("Q2_K", "[ATTENTION] Expérimental 2 bits (dégradation attendue sur 1B — préférer Q3_K_M ou Q4_K_M)", True),
]

DOSSIER_TRAVAIL = Path("travail_gguf_kouman_edge_1b")
DOSSIER_MODELE_BF16 = DOSSIER_TRAVAIL / "modele_bf16"
DOSSIER_LLAMA_CPP = DOSSIER_TRAVAIL / "llama.cpp"
DOSSIER_GGUF = DOSSIER_TRAVAIL / "gguf_export"
DOSSIER_TRAVAIL.mkdir(parents=True, exist_ok=True)
DOSSIER_GGUF.mkdir(parents=True, exist_ok=True)

client_hf = HfApi(token=JETON_HF)
print(f"Utilisateur Connecté sur HF : {client_hf.whoami()['name']}")
print(f"Dépôt cible     : https://huggingface.co/{DEPOT_MODELE}")
print(f"Cœurs CPU       : {os.cpu_count() or 2}")


def executer_cmd(cmd: list[str], cwd: Path | None = None) -> None:
    print("", " ".join(str(x) for x in cmd))
    res = subprocess.run(cmd, cwd=str(cwd) if cwd else None, text=True)
    if res.returncode != 0:
        raise RuntimeError(f"[ERREUR] Échec commande (code {res.returncode}) : {' '.join(str(x) for x in cmd)}")

## 2. Compilation Rapide de `llama.cpp` (`< 1 min` sur Colab)

In [ ]:
if shutil.which("cmake") is None:
    executer_cmd([sys.executable, "-m", "pip", "install", "--quiet", "cmake", "ninja"])

if not DOSSIER_LLAMA_CPP.exists():
    executer_cmd(["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp.git", str(DOSSIER_LLAMA_CPP)])

DOSSIER_BUILD = DOSSIER_LLAMA_CPP / "build"
nb_coeurs = str(max(2, os.cpu_count() or 2))
bin_quantize = DOSSIER_BUILD / "bin" / "llama-quantize"
bin_imatrix = DOSSIER_BUILD / "bin" / "llama-imatrix"
bin_cli = DOSSIER_BUILD / "bin" / "llama-cli"

if not (bin_quantize.exists() and bin_imatrix.exists() and bin_cli.exists()):
    executer_cmd([
        "cmake", "-B", str(DOSSIER_BUILD), "-S", str(DOSSIER_LLAMA_CPP),
        "-DCMAKE_BUILD_TYPE=Release",
        "-DLLAMA_CURL=OFF",
        "-DBUILD_SHARED_LIBS=OFF",
    ])
    executer_cmd([
        "cmake", "--build", str(DOSSIER_BUILD),
        "--config", "Release",
        "--target", "llama-quantize", "llama-imatrix", "llama-cli",
        "-j", nb_coeurs,
    ])

for binaire in (bin_quantize, bin_imatrix, bin_cli):
    if not binaire.exists():
        raise FileNotFoundError(f"[ERREUR] Binaire llama.cpp manquant : {binaire}")
print(f"[OK] Binaires llama.cpp prêts ({nb_coeurs} cœurs CPU).")

## 3. Téléchargement de `Tree-AI-lab/kouman-edge-1b` (Instruct BF16) & Conversion GGUF `F16` Maître

In [ ]:
if not (DOSSIER_MODELE_BF16 / "config.json").exists():
    print(f"Téléchargement du modèle Instruct BF16 depuis {DEPOT_MODELE}...")
    snapshot_download(
        repo_id=DEPOT_MODELE,
        repo_type="model",
        local_dir=str(DOSSIER_MODELE_BF16),
        ignore_patterns=["*.gguf", "*.imatrix"],
        token=JETON_HF,
    )

assert (DOSSIER_MODELE_BF16 / "config.json").exists(), "[ERREUR] config.json manquant dans le modèle téléchargé !"
fichiers_st = list(DOSSIER_MODELE_BF16.glob("*.safetensors"))
assert fichiers_st, "[ERREUR] Aucun fichier .safetensors trouvé dans Tree-AI-lab/kouman-edge-1b !"
print(f"[OK] Modèle Instruct BF16 vérifié ({len(fichiers_st)} fichier(s) safetensors).")

chemin_gguf_f16 = DOSSIER_GGUF / f"{NOM_COURT}-F16.gguf"
script_conv = DOSSIER_LLAMA_CPP / "convert_hf_to_gguf.py"
if not (chemin_gguf_f16.exists() and chemin_gguf_f16.stat().st_size > 500_000_000):
    executer_cmd([
        sys.executable, str(script_conv),
        str(DOSSIER_MODELE_BF16),
        "--outfile", str(chemin_gguf_f16),
        "--outtype", "f16",
    ])
assert chemin_gguf_f16.exists() and chemin_gguf_f16.stat().st_size > 500_000_000, "[ERREUR] GGUF F16 invalide !"
print(f"[OK] GGUF F16 maître prêt : {chemin_gguf_f16.stat().st_size / (1024**3):.2f} Go")

## 4. Calibration `imatrix` 4 Langues (`Baoulé + Dioula + Français + Anglais`)

In [ ]:
chemin_imatrix = DOSSIER_GGUF / f"{NOM_COURT}.imatrix"
if UTILISER_IMATRIX and not (chemin_imatrix.exists() and chemin_imatrix.stat().st_size > 100_000):
    chemin_calib = DOSSIER_TRAVAIL / "calibration_imatrix_4langues.txt"
    blocs_calib = []
    for nom_split in ("validation", "train"):
        p_parq = Path(hf_hub_download(
            repo_id=DEPOT_DATASET_CALIB,
            filename=f"data/{nom_split}-00000-of-00001.parquet",
            repo_type="dataset",
            token=JETON_HF,
        ))
        tbl = pq.read_table(p_parq, columns=["messages"])
        for conv in tbl["messages"].to_pylist():
            texte_conv = "\n".join(m["content"] for m in conv if isinstance(m, dict) and m.get("content"))
            if texte_conv:
                blocs_calib.append(texte_conv)
            if len(blocs_calib) >= 320:
                break
        if len(blocs_calib) >= 320:
            break
    if not blocs_calib:
        raise RuntimeError("[ERREUR] Corpus de calibration imatrix vide !")
    chemin_calib.write_text("\n\n".join(blocs_calib), encoding="utf-8")
    print(f"Calcul imatrix 4 langues ({len(blocs_calib)} dialogues, {CHUNKS_IMATRIX} chunks)...")
    executer_cmd([
        str(bin_imatrix),
        "-m", str(chemin_gguf_f16),
        "-f", str(chemin_calib),
        "-o", str(chemin_imatrix),
        "--chunks", str(CHUNKS_IMATRIX),
        "-t", nb_coeurs,
        "-ngl", "0",
    ])
    assert chemin_imatrix.exists() and chemin_imatrix.stat().st_size > 100_000, "[ERREUR] Fichier imatrix invalide !"
    print(f"[OK] imatrix 4 langues prête : {chemin_imatrix.stat().st_size / (1024**2):.1f} Mo")
elif chemin_imatrix.exists():
    print(f"[OK] imatrix déjà calculée : {chemin_imatrix.stat().st_size / (1024**2):.1f} Mo")

## 5. Quantification Complète GGUF (`Q8_0` → `Q2_K`)

In [ ]:
inventaire_gguf = []
for q_type, q_desc, q_avec_imatrix in QUANTIFICATIONS_GGUF:
    nom_fic = f"{NOM_COURT}-{q_type}.gguf"
    chemin_out = DOSSIER_GGUF / nom_fic
    cmd_q = [str(bin_quantize)]
    if UTILISER_IMATRIX and q_avec_imatrix and chemin_imatrix.exists():
        cmd_q.extend(["--imatrix", str(chemin_imatrix)])
    cmd_q.extend([str(chemin_gguf_f16), str(chemin_out), q_type, nb_coeurs])
    executer_cmd(cmd_q)
    taille_octets = chemin_out.stat().st_size
    assert taille_octets > 200_000_000, f"[ERREUR] Fichier GGUF anormalement petit : {nom_fic} ({taille_octets} octets)"
    inventaire_gguf.append({
        "fichier": nom_fic,
        "quant": q_type,
        "taille_mo": round(taille_octets / (1024 ** 2), 1),
        "taille_go": round(taille_octets / (1024 ** 3), 2),
        "imatrix": bool(UTILISER_IMATRIX and q_avec_imatrix),
        "usage": q_desc,
    })
    print(f"   [OK] {nom_fic:30s} : {taille_octets / (1024**2):7.1f} Mo")

if not GARDER_GGUF_F16:
    chemin_gguf_f16.unlink(missing_ok=True)
    print("F16 temporaire supprimé (le modèle Instruct BF16 est déjà dans le dépôt HF).")
else:
    t_f16 = chemin_gguf_f16.stat().st_size
    inventaire_gguf.insert(0, {
        "fichier": chemin_gguf_f16.name,
        "quant": "F16",
        "taille_mo": round(t_f16 / (1024 ** 2), 1),
        "taille_go": round(t_f16 / (1024 ** 3), 2),
        "imatrix": False,
        "usage": "Pleine précision FP16 GGUF",
    })

## 6. Tests Approfondis `llama.cpp` Multi-Quants (`Q8_0`, `Q4_K_M`, `Q3_K_M`, `Q2_K` — 8 Cas Réels)

In [ ]:
import re
import time

# Compile llama-completion (2 secondes car libllama est déjà compilé) : zéro bannière ASCII !
bin_completion = DOSSIER_BUILD / "bin" / "llama-completion"
if not bin_completion.exists():
    subprocess.run(
        [
            "cmake", "--build", str(DOSSIER_BUILD),
            "--config", "Release",
            "--target", "llama-completion",
            "-j", nb_coeurs,
        ],
        capture_output=True,
        text=True,
    )

binaire_test = bin_completion if bin_completion.exists() else bin_cli
mode_completion = binaire_test.name == "llama-completion"
print(f"Binaire de test actif : {binaire_test.name} (mode_completion={mode_completion})")

TESTS_APPROFONDIS = [
    ("1. Identité croisée (OOD)", "bonjour je suis nicanor et toi ?"),
    ("2. Identité Edge / Mobile", "quelle version de Kouman es-tu ?"),
    ("3. Dioula natif (salutation)", "I ni sɔgɔma"),
    ("4. Dioula libre (santé)", "n b'a fɛ i ka kuma n ye kɛnɛya kan"),
    ("5. Baoulé libre (village)", "kan klɔ su mɛn dilɛ’n i ndɛ kle min"),
    ("6. Bascule FR -> DYU", "bonjour repond moi en dioula"),
    ("7. Traduction FR -> BCI", "traduis en baoulé : la vie est belle au village"),
    ("8. Traduction FR -> DYU", "traduis en dioula : ma sœur prépare du riz chaud pour le dîner de la famille"),
]

# Choisis les quants à comparer (ajoute/retire selon ton temps dispo)
QUANTS_A_TESTER = ["Q8_0", "Q4_K_M", "Q3_K_M", "Q2_K"]


def nettoyer_sortie_llama(texte: str, question: str) -> str:
    """Supprime la bannière ASCII llama.cpp, les lignes de statut et les balises de fin."""
    lignes_propres = []
    for ligne in texte.splitlines():
        l_strip = ligne.strip()
        if not l_strip:
            if lignes_propres and lignes_propres[-1] != "":
                lignes_propres.append("")
            continue
        if any(c in l_strip for c in ("▄", "█", "▀")):
            continue
        if re.match(r"^(Loading model|build\s*:|model\s*:|modalities\s*:|available commands|/exit|/regen|/clear|/read|\[ Prompt:|Exiting|EOF by user)", l_strip):
            continue
        if l_strip.startswith("> ") and question[:20].lower() in l_strip.lower():
            continue
        if l_strip == ">":
            continue
        lignes_propres.append(ligne.rstrip())
    res = "\n".join(lignes_propres).strip()
    for balise in ("<|eot_id|>", "<|end_of_text|>", "[end of text]"):
        res = res.replace(balise, "")
    return res.strip()


for q_test in QUANTS_A_TESTER:
    fic_test = DOSSIER_GGUF / f"{NOM_COURT}-{q_test}.gguf"
    if not fic_test.exists():
        print(f"[ATTENTION] Fichier absent : {fic_test.name}")
        continue
    taille_mo = fic_test.stat().st_size / (1024 ** 2)
    print("\n" + "═" * 78)
    print(f"BENCHMARK QUALITATIF GGUF : {fic_test.name} ({taille_mo:.0f} Mo)")
    print("═" * 78)
    for titre_t, q_t in TESTS_APPROFONDIS:
        t0 = time.perf_counter()
        if mode_completion:
            # llama.cpp ajoute automatiquement <|begin_of_text|> (BOS) selon les métadonnées GGUF
            prompt_t = (
                "<|start_header_id|>user<|end_header_id|>\n\n"
                f"{q_t}<|eot_id|>"
                "<|start_header_id|>assistant<|end_header_id|>\n\n"
            )
            cmd_test = [
                str(binaire_test),
                "-m", str(fic_test),
                "-p", prompt_t,
                "-n", "180",
                "--temp", "0.35",
                "--top-p", "0.9",
                "--repeat-penalty", "1.08",
                "--seed", "42",
                "-t", nb_coeurs,
                "--no-display-prompt",
                "-no-cnv",
                "--no-perf",
            ]
        else:
            cmd_test = [
                str(binaire_test),
                "-m", str(fic_test),
                "-p", q_t,
                "-n", "180",
                "--temp", "0.35",
                "--top-p", "0.9",
                "--repeat-penalty", "1.08",
                "--seed", "42",
                "-t", nb_coeurs,
                "--single-turn",
                "--simple-io",
                "--no-perf",
            ]
        res = subprocess.run(
            cmd_test,
            stdin=subprocess.DEVNULL,
            capture_output=True,
            text=True,
            timeout=180,
        )
        duree = time.perf_counter() - t0
        rep = nettoyer_sortie_llama(res.stdout or "", q_t)
        if not rep and res.stderr:
            rep = f"[STDERR] {res.stderr.strip()[-300:]}"
        print(f"\n── {titre_t} ({duree:.1f}s)")
        print(f"Utilisateur : {q_t}")
        print(f"Assistant : {rep}")

## 7. Génération de la Model Card (`README.md`) & Publication dans `Tree-AI-lab/kouman-edge-1b`

In [ ]:
lignes_tableau = [
    "| Fichier GGUF | Quant | Taille | `imatrix` | Usage recommandé |",
    "|---|---|---:|:---:|---|",
]
for item in inventaire_gguf:
    taille_txt = f"{item['taille_go']:.2f} Go" if item["taille_mo"] >= 1000 else f"{item['taille_mo']:.0f} Mo"
    badge_im = "[OK]" if item["imatrix"] else "—"
    lignes_tableau.append(f"| `{item['fichier']}` | `{item['quant']}` | {taille_txt} | {badge_im} | {item['usage']} |")

readme_md = "\n".join([
    "---",
    "license: llama3.2",
    "base_model: meta-llama/Llama-3.2-1B",
    "language:",
    "- bci",
    "- dyu",
    "- fr",
    "- en",
    "pipeline_tag: text-generation",
    "tags:",
    "- kouman-ai",
    "- tree-ai-lab",
    "- cote-divoire",
    "- baoule",
    "- dioula",
    "- edge-ai",
    "- mobile",
    "- gguf",
    "- llama-cpp",
    "---",
    "",
    "# Kouman Edge 1B (`Tree-AI-lab/kouman-edge-1b`)",
    "",
    "**Kouman Edge 1B** est le modèle d'intelligence artificielle **mobile & hors-ligne (on-device)** développé par **Tree AI Lab** (Côte d'Ivoire) sur une architecture **Llama 3.2 1B** (`1,24B` paramètres). Il parle et traduit **4 langues** : **Baoulé (`bci`)**, **Dioula (`dyu`)**, **Français (`fr`)** et **Anglais (`en`)**.",
    "",
    "Ce dépôt réunit au même endroit :",
    "- Le **modèle Instruct 1B complet en `bfloat16`** (`safetensors`, natif `transformers` / `vLLM` / `Ollama`, zéro `trust_remote_code`),",
    "- Toute la gamme **GGUF `llama.cpp` de `Q8_0` jusqu'à `Q2_K`**, calibrée avec une **`imatrix` 4 langues** pour préserver le Baoulé et le Dioula sur smartphone.",
    "",
    "## Performances CPT (`~1,70B` tokens, 6 488 steps WSD)",
    "",
    "| Langue | Base Llama 3.2 1B (PPL) | **Kouman Edge 1B CPT (PPL)** | Gain |",
    "|---|---:|---:|---:|",
    "| Baoulé (`bci`) | 20.74 | **2.90** | **-17.84** (÷ 7,2×) |",
    "| Dioula (`dyu`) | 19.58 | **2.25** | **-17.33** (÷ 8,7×) |",
    "| Français (`fr`) | 9.33 | **4.40** | **-4.93** (0 % oubli) |",
    "| Anglais (`en`) | 6.77 | **5.12** | **-1.65** (0 % oubli) |",
    "",
    "## Fichiers GGUF disponibles (`llama.cpp` • `Q8_0` → `Q2_K`)",
    "",
    *lignes_tableau,
    "",
    "## Utilisation rapide",
    "",
    "### Sur smartphone (Android / iOS hors-ligne)",
    "- Téléchargez **`kouman-edge-1b-Q4_K_M.gguf`** (recommandé) ou **`kouman-edge-1b-Q3_K_M.gguf`** dans **PocketPal AI**, **ChatterUI** ou toute app basée sur `llama.cpp`.",
    "",
    "### Avec `llama.cpp`",
    "```bash",
    f'llama-cli -hf {DEPOT_MODELE}:Q4_K_M -cnv -p "Tu es Kouman Edge 1B, assistant IA de Tree AI Lab."',
    "```",
    "",
    "### Avec `transformers` (modèle Instruct BF16)",
    "```python",
    "import torch",
    "from transformers import AutoModelForCausalLM, AutoTokenizer",
    "",
    f"repo = {DEPOT_MODELE!r}",
    "tok = AutoTokenizer.from_pretrained(repo)",
    "mod = AutoModelForCausalLM.from_pretrained(repo, torch_dtype=torch.bfloat16, device_map='auto')",
    "prompt = tok.apply_chat_template([{'role': 'user', 'content': 'I ni sɔgɔma !'}], tokenize=False, add_generation_prompt=True)",
    "ids = tok(prompt, return_tensors='pt', add_special_tokens=False).to(mod.device)",
    "out = mod.generate(**ids, max_new_tokens=256)",
    "print(tok.decode(out[0, ids['input_ids'].shape[1]:], skip_special_tokens=True))",
    "```",
]) + "\n"
(DOSSIER_GGUF / "README.md").write_text(readme_md, encoding="utf-8")

if PUBLIER_SUR_HF:
    print(f"Upload des {len(inventaire_gguf)} fichiers GGUF + imatrix + README.md vers {DEPOT_MODELE}...")
    client_hf.upload_folder(
        folder_path=str(DOSSIER_GGUF),
        repo_id=DEPOT_MODELE,
        repo_type="model",
        commit_message="feat: gamme complète GGUF llama.cpp (Q8_0 -> Q2_K, imatrix 4 langues) + Model Card",
    )
    print(f"TERMINÉ ! Dépôt complet (Instruct BF16 + GGUF Q8_0->Q2_K) : https://huggingface.co/{DEPOT_MODELE}")